# Greenhouse Quality Monitoring System (Fuzzy Logic)

Converted from `main.py` into a Google Colab-compatible Jupyter Notebook.

The original program supports interactive input and three preset simulation/verification options.

In [ ]:
# Imports from the original project
from fuzzy_controller import compute_gqi
from actuator_control import decide_actuators


In [ ]:
def print_report(temperature, humidity, soil_moisture):
    result = compute_gqi(temperature, humidity, soil_moisture)

    print("=" * 60)
    print(f"INPUTS: Temperature = {temperature} C | "
          f"Humidity = {humidity} %RH | Soil Moisture = {soil_moisture} %")
    print("-" * 60)

    print("Membership values:")
    print("  Temperature :", {k: round(v, 3) for k, v in result["temp_mf"].items()})
    print("  Humidity    :", {k: round(v, 3) for k, v in result["hum_mf"].items()})
    print("  Soil Moist. :", {k: round(v, 3) for k, v in result["soil_mf"].items()})

    print("\nFired rules and firing strengths:")
    if result["fired_rules"]:
        for r in result["fired_rules"]:
            print(f"  {r['rule_id']:>4}: Temp={r['temp_label']:<7} "
                  f"Hum={r['hum_label']:<8} Soil={r['soil_label']:<9} "
                  f"-> GQI={r['output_label']:<10} "
                  f"w = min({r['mu_temp']:.3f}, {r['mu_hum']:.3f}, {r['mu_soil']:.3f}) "
                  f"= {r['firing_strength']:.3f}")
    else:
        print("  No rules fired (inputs outside all membership ranges).")

    gqi = result["gqi"]
    status = result["status"]
    print(f"\nFinal GQI (weighted average) = {gqi:.2f} %" if gqi is not None
          else "\nFinal GQI could not be computed.")
    print(f"GQI Status = {status}")

    actuators = decide_actuators(gqi, soil_moisture)
    print("\nActuator response:")
    print(f"  Fan   : {actuators['fan']}")
    print(f"  Pump  : {actuators['pump']}")
    print(f"  Alarm : {actuators['alarm']}")
    print("=" * 60 + "\n")


In [ ]:
def run_simulation_examples():
    print("\nRunning preset simulation examples from the report...\n")
    examples = [
        (20, 50, 50),   # Example 1
        (40, 80, 80),   # Example 2
        (32, 45, 35),   # Example 3 - must match report: GQI = 50%
    ]

    for i, (t, h, s) in enumerate(examples, start=1):
        print(f"### Example {i} ###")
        print_report(t, h, s)


In [ ]:
def run_interactive():
    print("\n--- Greenhouse Quality Index (GQI) Calculator ---")
    try:
        t = float(input("Enter Temperature (0-50 C): "))
        h = float(input("Enter Humidity (0-100 %RH): "))
        s = float(input("Enter Soil Moisture (0-100 %): "))
    except ValueError:
        print("Please enter numeric values only.")
        return
    print_report(t, h, s)


In [ ]:
def main():
    print("Greenhouse Quality Monitoring System (Fuzzy Logic)")
    print("1. Interactive mode (enter your own values)")
    print("2. Simulation mode (run report's 3 preset examples)")
    print("3. Verify report example (32, 45, 35 -> should give GQI = 50%)")
    choice = input("Choose an option (1/2/3): ").strip()

    if choice == "1":
        run_interactive()
    elif choice == "2":
        run_simulation_examples()
    elif choice == "3":
        print_report(32, 45, 35)
    else:
        print("Invalid choice. Running the report verification example instead.")
        print_report(32, 45, 35)


## Run the program

Run the next cell and choose:
- `1` for interactive mode
- `2` for the three preset simulations
- `3` for the report verification example

In [ ]:
main()


### Important for Google Colab

The original `main.py` imports `fuzzy_controller` and `actuator_control`. Those modules must also be uploaded to the Colab session (or placed in the same project folder) before running the notebook.